## Dataset

For this homework we will be using the Yellow 2025-11 data from the official website:

```bash
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [1]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[*]")\
        .appName("HW06")\
            .getOrCreate()

spark.version

26/03/07 22:20:23 WARN Utils: Your hostname, Khangs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.1.37 instead (on interface en0)
26/03/07 22:20:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 22:20:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.5'

In [5]:
sf = spark.read.parquet('yellow_tripdata_2025-11.parquet')
sf.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [13]:
sf.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB

In [ ]:
sf.repartition(4).write.parquet('file_repartitioned',mode="overwrite")

In [8]:
import os

folder_path = 'file_repartitioned'

# Get all .parquet files inside the directory
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

total_size_bytes = sum(os.path.getsize(os.path.join(folder_path, f)) for f in parquet_files)
num_files = len(parquet_files)

if num_files > 0:
    avg_size_mb = (total_size_bytes / num_files) / (1024 * 1024)
    print(f"Number of Parquet files: {num_files}")
    print(f"Average size: {avg_size_mb:.2f} MB")
else:
    print("No Parquet files found.")


Number of Parquet files: 4
Average size: 24.42 MB


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604
- 225,768

In [25]:
sf.filter((sf['tpep_pickup_datetime'] >= '2025-11-15') & (sf['tpep_pickup_datetime'] < '2025-11-16')).count()

162604

## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- 90.6
- 134.5

In [39]:
from pyspark.sql import functions as F

sf = sf.withColumn(
    "duration_hours",
    F.round(    
    (F.unix_timestamp("tpep_dropoff_datetime") - 
     F.unix_timestamp("tpep_pickup_datetime")) / 3600,1))
sf.select(['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'duration_hours'])\
    .orderBy('duration_hours',ascending=False).show(5)

+--------------------+---------------------+--------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|duration_hours|
+--------------------+---------------------+--------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|          90.6|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|          76.9|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|          76.2|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|          69.3|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|          67.1|
+--------------------+---------------------+--------------+
only showing top 5 rows



## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- 4040
- 8080

In [40]:
spark.sparkContext.uiWebUrl

'http://192.168.1.37:4040'

# Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- Governor's Island/Ellis Island/Liberty Island
- Arden Heights
- Rikers Island
- Jamaica Bay

In [50]:
taxi = spark.read.csv('taxi_zone_lookup.csv', header=True, inferSchema=True)
taxi.createOrReplaceTempView('taxi_zone_lookup')
taxi.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [54]:
location = sf.select('PULocationID')
location.join(taxi, location['PULocationID'] == taxi['LocationID'])\
    .groupby('Zone').count()\
        .withColumnRenamed('count', 'count_pickup')\
            .orderBy('count_pickup').show(5)

+--------------------+------------+
|                Zone|count_pickup|
+--------------------+------------+
|Governor's Island...|           1|
|Eltingville/Annad...|           1|
|       Arden Heights|           1|
|       Port Richmond|           3|
|       Rikers Island|           4|
+--------------------+------------+
only showing top 5 rows

